In [ ]:
import cv2
import torch
import requests
import numpy as np
from pathlib import Path
from zipfile import ZipFile
from PIL import Image
!pip install torchmetrics
from torchmetrics import Accuracy
from torchvision.models import EfficientNet_B0_Weights, efficientnet_b0
from torch.utils.data import DataLoader, random_split
from tqdm import tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 27.7 MB/s eta 0:00:00


In [ ]:
url = "https://data.mendeley.com/public-files/datasets/8fmvr9m98w/files/9dcdeaa8-4df1-4eb9-98df-3231f66a10c7/file_downloaded"
root_path   = Path("root/")
data_path   = root_path / "data/"
model_path  = root_path / "lstm_model.pth"
file_path   = data_path / "File.zip"
videos_path = data_path / "videos"

if videos_path.is_dir():
    print("Data already there!")
else:
    videos_path.mkdir(exist_ok=True, parents=True)
    if not file_path.exists():
        print("Downloading Data …")
        with open(file_path, "wb") as f:
            f.write(requests.get(url).content)
        print("Downloaded.")
    with ZipFile(file_path, "r") as z:
        print("Extracting …")
        z.extractall(data_path)
        print("Extracted.")


Downloaded.
Extracting …
Extracted.


In [ ]:
NUM_FRAMES = 16

weights        = EfficientNet_B0_Weights.DEFAULT
auto_transform = weights.transforms()


In [ ]:
def get_frames(vid_path: str, n_frames: int = NUM_FRAMES) -> torch.Tensor:
    """Return a (n_frames, C, H, W) tensor of uniformly sampled frames."""
    cap    = cv2.VideoCapture(vid_path)
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0: # corrupt video
        total = 1
    indices = np.linspace(0, max(total - 1, 0), n_frames, dtype=int)
    idx_set = set(indices.tolist())
    frames  = {}
    fid     = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if fid in idx_set:
            img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            frames[fid] = auto_transform(img)
        fid += 1
    cap.release()


    result = [frames.get(i, list(frames.values())[-1]) for i in indices]
    return torch.stack(result)   # (n_frames, C, H, W)



In [ ]:
class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, data_path: Path):
        super().__init__()
        self.files = sorted(data_path.glob("*/*.avi"))
        classes = sorted({f.parent.stem for f in self.files})
        self.class_to_idx = {c: i for i, c in enumerate(classes)}
        self.classes      = classes

    def __len__(self):
        return len(self.files)

    def __getitem__(self, index):
        path  = self.files[index]
        label = self.class_to_idx[path.parent.stem]
        return get_frames(str(path)), label   # (16, C, H, W), int


In [ ]:
full_dataset = CustomDataset(data_path / "ASL_dynamic")
classes      = full_dataset.class_to_idx
n_classes    = len(classes)
print(f"Classes ({n_classes}): {classes}")

val_size   = max(1, int(0.2 * len(full_dataset)))
train_size = len(full_dataset) - val_size
train_dataset, val_dataset = random_split(
    full_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)
print(f"Train: {train_size}  |  Val: {val_size}")
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=8, shuffle=False,
                          num_workers=2, pin_memory=True)



Classes (31): {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6, 'H': 7, 'HELLO': 8, 'I': 9, 'J': 10, 'K': 11, 'L': 12, 'M': 13, 'N': 14, 'NO': 15, 'O': 16, 'P': 17, 'Q': 18, 'R': 19, 'S': 20, 'SORRY': 21, 'T': 22, 'THANKYOU': 23, 'U': 24, 'V': 25, 'W': 26, 'X': 27, 'Y': 28, 'YES': 29, 'Z': 30}
Train: 240  |  Val: 60


In [ ]:
class EarlyStop:
    def __init__(self, patience: int = 7, tolerance: float = 1e-4):
        self.patience  = patience
        self.tolerance = tolerance
        self.counter   = 0
        self.best_loss = None

    def stop(self, loss: float) -> bool:
        if self.best_loss is None:
            self.best_loss = loss
            return False
        if loss < self.best_loss - self.tolerance:
            self.best_loss = loss
            self.counter   = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                return True
        return False

In [ ]:
class LSTM_CLASSIFIER(torch.nn.Module):
    def __init__(self, num_classes: int, lstm_hidden: int = 512,
                 lstm_layers: int = 2, dropout: float = 0.4):
        super().__init__()

        backbone      = efficientnet_b0(weights=weights)
        cnn_features  = backbone.features
        adaptive_pool = torch.nn.AdaptiveAvgPool2d(1)
        flatten       = torch.nn.Flatten()

        projection    = torch.nn.Sequential(
            torch.nn.Linear(1280, 512),
            torch.nn.BatchNorm1d(512),
            torch.nn.ReLU(inplace=True),
            torch.nn.Dropout(dropout),
        )
        self.image_embedding = torch.nn.Sequential(
            cnn_features,
            adaptive_pool,
            flatten,
            projection,
        )


        for param in cnn_features.parameters():
            param.requires_grad = False


        self.lstm = torch.nn.LSTM(
            input_size=512,
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            dropout=dropout if lstm_layers > 1 else 0.0,
        )


        self.head = torch.nn.Sequential(
            torch.nn.Linear(lstm_hidden, 256),
            torch.nn.BatchNorm1d(256),
            torch.nn.ReLU(inplace=True),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(256, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C, H, W = x.shape

        x_flat = x.view(B * T, C, H, W)
        emb    = self.image_embedding(x_flat)
        emb    = emb.view(B, T, -1)

        _, (h_n, _) = self.lstm(emb)
        out = self.head(h_n[-1])
        return out



In [ ]:
device   = "cuda" if torch.cuda.is_available() else "cpu"
model    = LSTM_CLASSIFIER(n_classes).to(device)
loss_fn  = torch.nn.CrossEntropyLoss(label_smoothing=0.1)
acc_fn   = Accuracy(task="multiclass", num_classes=n_classes).to(device)


optimiser = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=3e-4, weight_decay=1e-4
)
epochs    = 20
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=epochs)


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 154MB/s]


In [ ]:
early_stopper = EarlyStop(patience=7, tolerance=1e-3)
best_val_acc  = 0.0

for epoch in range(1, epochs + 1):
    print(f"\n{'='*60}")
    print(f"Epoch {epoch}/{epochs}  |  LR: {scheduler.get_last_lr()[0]:.2e}")
    print(f"{'='*60}")

    #Train
    model.train()
    total_loss = total_acc = 0.0
    for data, labels in tqdm(train_loader, desc="Training"):
        data, labels = data.to(device), labels.to(device)
        optimiser.zero_grad()
        logits = model(data)
        loss   = loss_fn(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimiser.step()
        total_loss += loss.item()
        total_acc  += acc_fn(logits.argmax(1), labels).item()

    train_loss = total_loss / len(train_loader)
    train_acc  = total_acc  / len(train_loader) * 100

    # Validate
    model.eval()
    val_loss = val_acc = 0.0
    with torch.inference_mode():
        for data, labels in tqdm(val_loader, desc="Validating"):
            data, labels = data.to(device), labels.to(device)
            logits   = model(data)
            val_loss += loss_fn(logits, labels).item()
            val_acc  += acc_fn(logits.argmax(1), labels).item()

    val_loss /= len(val_loader)
    val_acc   = val_acc / len(val_loader) * 100

    print(f"Train --  loss: {train_loss:.4f}  |  acc: {train_acc:.2f}%")
    print(f"Val   --  loss: {val_loss:.4f}  |  acc: {val_acc:.2f}%")

    scheduler.step()

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), model_path)
        print(f"  New best val acc {best_val_acc:.2f}% — model saved.")

    if early_stopper.stop(val_loss):
        print("Early stopping triggered.")
        break

print(f"\nTraining complete. Best val acc: {best_val_acc:.2f}%")
print(f"Weights saved to: {model_path}")



Epoch 1/20  |  LR: 3.00e-04


Validating: 100%|██████████| 8/8 [00:11<00:00,  1.45s/it]


Train  →  loss: 3.2505  |  acc: 12.08%
Val    →  loss: 3.2832  |  acc: 35.94%
  ✓ New best val acc 35.94% — model saved.

Epoch 2/20  |  LR: 2.98e-04


Validating: 100%|██████████| 8/8 [00:13<00:00,  1.72s/it]


Train  →  loss: 2.4031  |  acc: 50.00%
Val    →  loss: 2.1746  |  acc: 64.06%
  ✓ New best val acc 64.06% — model saved.

Epoch 3/20  |  LR: 2.93e-04


...

Train  →  loss: 0.7399  |  acc: 99.58%
Val    →  loss: 0.8115  |  acc: 98.44%

Epoch 19/20  |  LR: 7.34e-06


Validating: 100%|██████████| 8/8 [00:10<00:00,  1.35s/it]


Train  →  loss: 0.7401  |  acc: 100.00%
Val    →  loss: 0.8080  |  acc: 98.44%

Epoch 20/20  |  LR: 1.85e-06


Validating: 100%|██████████| 8/8 [00:11<00:00,  1.42s/it]

Train  →  loss: 0.7432  |  acc: 99.58%
Val    →  loss: 0.8096  |  acc: 98.44%

Training complete. Best val acc: 98.44%
Weights saved to: root/lstm_model.pth
